In [2]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [16]:
data = pd.read_csv('./dataset/HR.csv')

In [17]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14999 entries, 0 to 14998
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   satisfaction_level     14999 non-null  float64
 1   last_evaluation        14999 non-null  float64
 2   number_project         14999 non-null  int64  
 3   average_montly_hours   14999 non-null  int64  
 4   time_spend_company     14999 non-null  int64  
 5   Work_accident          14999 non-null  int64  
 6   left                   14999 non-null  int64  
 7   promotion_last_5years  14999 non-null  int64  
 8   part                   14999 non-null  object 
 9   salary                 14999 non-null  object 
dtypes: float64(2), int64(6), object(2)
memory usage: 1.1+ MB


In [18]:
data.part.unique()

array(['sales', 'accounting', 'hr', 'technical', 'support', 'management',
       'IT', 'product_mng', 'marketing', 'RandD'], dtype=object)

In [19]:
data.salary.unique()

array(['low', 'medium', 'high'], dtype=object)

In [20]:
# 对离散字符串的两种处理方式 ， 1、转成数字  2、one-hot编码
data = data.join(pd.get_dummies(data.part, dtype=int).join(pd.get_dummies(data.salary, dtype=int)))

In [21]:
data

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,part,salary,...,hr,management,marketing,product_mng,sales,support,technical,high,low,medium
0,0.38,0.53,2,157,3,0,1,0,sales,low,...,0,0,0,0,1,0,0,0,1,0
1,0.80,0.86,5,262,6,0,1,0,sales,medium,...,0,0,0,0,1,0,0,0,0,1
2,0.11,0.88,7,272,4,0,1,0,sales,medium,...,0,0,0,0,1,0,0,0,0,1
3,0.72,0.87,5,223,5,0,1,0,sales,low,...,0,0,0,0,1,0,0,0,1,0
4,0.37,0.52,2,159,3,0,1,0,sales,low,...,0,0,0,0,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14994,0.40,0.57,2,151,3,0,1,0,support,low,...,0,0,0,0,0,1,0,0,1,0
14995,0.37,0.48,2,160,3,0,1,0,support,low,...,0,0,0,0,0,1,0,0,1,0
14996,0.37,0.53,2,143,3,0,1,0,support,low,...,0,0,0,0,0,1,0,0,1,0
14997,0.11,0.96,6,280,4,0,1,0,support,low,...,0,0,0,0,0,1,0,0,1,0


In [25]:
# 把part salary列删除
data.drop(columns=['part', 'salary'], inplace=True)

KeyError: "['part', 'salary'] not found in axis"

In [26]:
data.left.value_counts()

left
0    11428
1     3571
Name: count, dtype: int64

In [27]:
Y_data = data.left.values.reshape(-1, 1)

In [28]:
Y = torch.from_numpy(Y_data).type(torch.FloatTensor)

In [34]:
X_data = data[[c for c in data.columns if c !='left']].values

In [35]:
X = torch.from_numpy(X_data).type(torch.FloatTensor)

In [36]:
X.shape

torch.Size([14999, 20])

In [53]:
# pytorch中最常用的一种创建模型的方式
# 子类的写法
from torch import nn

In [59]:
class HRModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin_1 = nn.Linear(20, 64)
        self.lin_2 = nn.Linear(64, 64)
        self.lin_3 = nn.Linear(64, 1)
        self.activate = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, input):
        # 定义前向传播
        x = self.lin_1(input)
        x = self.activate(x)
        x = self.lin_2(x)
        x = self.activate(x)
        x = self.lin_3(x)
        x = self.sigmoid(x)
        return x
        

In [60]:
lr = 0.001

In [64]:
# 定义获取模型的函数和优化器
def get_model():
    model = HRModel()
    return model, torch.optim.Adam(model.parameters(), lr=lr)

In [65]:
# 定义损失函数
loss_fn = nn.BCELoss()

In [66]:
model, opt = get_model()

In [67]:
batch_size = 64
steps = len(data) // 64
epochs = 100

In [68]:
# 训练过程
for epoch in range(epochs):
    for i in range(steps):
        start = i * batch_size
        end = start + batch_size
        x = X[start:end]
        y = Y[start:end]
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        opt.zero_grad()
        loss.backward()
        opt.step()


In [69]:
((model(X).data.numpy() > 0.5) == Y.numpy()).mean()

0.7619174611640777

In [70]:
### 使用dataset重构数据

In [71]:
# PyTorch中有一个Dataset类，可以把任意的具有__len__和__getitem__方法的对象包装成Dataset对象
# Dataset自动取数据

In [72]:
from torch.utils.data import TensorDataset

In [73]:
HRdataset = TensorDataset(X, Y)

In [74]:
HRdataset

In [75]:
model, opt = get_model()

In [77]:
# 重写训练过程
for epoch in range(epochs):
    for step in range(steps):
        # 取数据不一样了
        x, y = HRdataset[step * batch_size: step * batch_size + batch_size]
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        opt.zero_grad()
        loss.backward()
        opt.step()

In [79]:
### 使用DataLoader重构数据

In [81]:
# dataLoader可以自动分批取数据
# dataLoader是由dataset创建出来的
# 有了dataset就不需要按切片取数据
from torch.utils.data import DataLoader

In [82]:
HR_ds = TensorDataset(X, Y)

In [83]:
HR_dl = DataLoader(HR_ds, batch_size=batch_size)

In [84]:
model, opt = get_model()

In [85]:
for epoch in range(epochs):
    for x, y in HR_dl:
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        opt.zero_grad()
        loss.backward()
        opt.step()

In [86]:
### 添加校验

In [87]:
# 需要分割出训练数据和测试数据
# 需要分割出训练数据和测试数据
from sklearn.model_selection import train_test_split

In [88]:
# 切割数据 -> 分别创建训练数据和测试数据的DataLoader --> 训练过程  --> 校验过程

In [93]:
train_x , test_x, train_y, test_y = train_test_split(X_data, Y_data)

In [94]:
# 转成tensor
train_x = torch.from_numpy(train_x).type(torch.FloatTensor)
train_y = torch.from_numpy(train_y).type(torch.FloatTensor)

test_x = torch.from_numpy(test_x).type(torch.FloatTensor)
test_y = torch.from_numpy(test_y).type(torch.FloatTensor)

In [95]:
# 转成Dataset和DataLoader
train_ds = TensorDataset(train_x, train_y)
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

test_ds = TensorDataset(test_x, test_y)
test_dl = DataLoader(test_ds, batch_size=batch_size * 2, shuffle=True)

In [96]:
# 定义计算准确率的函数
def accutacy(out, yb):
    return ((out.data.numpy() > 0.5) == yb.numpy()).mean()

In [97]:
# pytorch 中有训练模式，和测试/推理模式  model.train() model.eval()
# 训练和推理测试模式对一些特殊层会有不同的表现，比如：dropout， bn等
model, pot = get_model()

In [99]:
for epoch in range(epochs):
    # 训练的时候，调到训练模式
    model.train()
    for xb, yb in train_dl:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()

    # 每训练100次输出一次测试结果
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            # 计算平均损失和准确率
            valib_loss = sum([loss_fn(model(x), y) for x, y in test_dl]) / len(test_dl)
            acc_mean = np.mean([accutacy(model(x), y) for x, y in test_dl])
        print(epoch, valib_loss, acc_mean)

0 tensor(1.0585) 0.7721491228070175
10 tensor(1.0681) 0.773999451754386
20 tensor(1.0742) 0.7721491228070175
30 tensor(1.0615) 0.771532346491228
40 tensor(1.0775) 0.7690652412280702
50 tensor(1.0649) 0.7727658991228069
60 tensor(1.0606) 0.7727658991228069
70 tensor(1.0773) 0.7733826754385965
80 tensor(1.0750) 0.7709155701754385
90 tensor(1.0499) 0.7702987938596491
